<table style="width:100%; border:none; border-collapse:collapse; background:transparent;">
  <tr style="border:none; background:transparent;">
    <td style="width:130px; border:none; vertical-align:middle;" align="center">
      <img alt="udeA logo" height="135px" src="https://github.com/freddyduitama/images/blob/master/logo.png?raw=true" style="width:95px; height:135px;">
    </td>
    <td style="border:none; vertical-align:middle; text-align:center;">
      <p><font color='#0B5345' size="5" face="Lucida Calligraphy,Comic Sans MS,Lucida Console"><b>Laboratorio No. 2: Vigenère en MIPS.</b></font></p>
      <p style="margin: 4px 0;"><font face="Courier New" size="3">Arquitectura de Computadores y Laboratorio · 2026-2</font></p>
      <p style="margin: 4px 0;"><font face="Courier New" size="3">Integrantes: Santiago Sánchez Valencia · Salomé Restrepo Lugo</font></p>
      <p style="margin: 8px 0 0 0;"><font color='#0B5345' face="Lucida Calligraphy,Comic Sans MS,Lucida Console" size="3"><b>Universidad de Antioquia.</b></font></p>
    </td>
  </tr>
</table>

### <font color='#000000' face="Courier New" size="5">1. Objetivo y descripción del problema.</font>

<p align="justify"><font face="Verdana" size='2'>
Este laboratorio consiste en implementar el algoritmo de cifrado y descifrado de Vigenère en lenguaje ensamblador MIPS de 32 bits, utilizando la herramienta MARS. El objetivo es construir un programa estructurado y modular que permita procesar mensajes mediante un menú interactivo en consola.
</font></p>

<p align="justify"><font face="Verdana" size='2'>
El funcionamiento del programa se basa en las siguientes condiciones:
<ul>
  <li align="justify"><font face="Verdana" size='2'><b>Entradas:</b> Para la opción de cifrar, el texto se lee desde un archivo llamado <code>input.txt</code>, el cual contiene texto en inglés y no supera los 2048 bytes. La clave secreta se le solicita directamente al usuario por consola durante la ejecución. Para descifrar, el programa lee el archivo <code>criptogram.txt</code> que se generó previamente.</font></li>
  <li align="justify"><font face="Verdana" size='2'><b>Salidas:</b> Al cifrar, el mensaje resultante se guarda en un archivo de texto llamado <code>criptogram.txt</code>. Al descifrar, el texto recuperado se guarda en un archivo llamado <code>deciphered.txt</code>.</font></li>
  <li align="justify"><font face="Verdana" size='2'><b>Herramientas:</b> Se utilizó el simulador MARS 4.5 sobre la arquitectura MIPS32, apoyándonos en los servicios del sistema (<i>syscalls</i>) para interactuar con la consola y gestionar la apertura, lectura, escritura y cierre de los archivos.</font></li>
</ul>
</font></p>

### <font color='#000000' face="Courier New" size="5">2. Fundamento matemático.</font>

<p align="justify"><font face="Verdana" size='2'>
El cifrado de Vigenère es un método de cifrado polialfabético donde cada carácter del mensaje original se desplaza según la posición de la letra correspondiente en la palabra clave. Como la clave suele ser más corta que el mensaje, esta se va repitiendo cíclicamente hasta cubrir todo el texto.
</font></p>

<p align="justify"><font face="Verdana" size='2'>
En nuestra solución decidimos trabajar con todos los <b>caracteres imprimibles de la tabla ASCII</b>, que van desde el espacio (ASCII 32) hasta el símbolo <code>~</code> (ASCII 126). Esto nos da un alfabeto de <b>95 caracteres posibles (L = 95)</b>, lo que permite que el programa no solo cifre letras minúsculas, sino también mayúsculas, números, signos de puntuación y espacios sin alterar el texto original.
</font></p>

$$C_i = [ (M_i - 32) + (K_j - 32) ] \bmod 95 + 32$$

<p align="justify"><font face="Verdana" size='2'>
<b>Para cifrar:</b> Tomamos el valor ASCII del carácter del mensaje (<code>Mᵢ</code>) y el de la clave (<code>Kⱼ</code>). Les restamos 32 para llevarlos a un rango de 0 a 94, los sumamos en módulo 95 y luego le sumamos 32 para volver a obtener el código ASCII del carácter cifrado.
</font></p>

$$M_i = [ (C_i - 32) - (K_j - 32) + 95 ] \bmod 95 + 32$$

<p align="justify"><font face="Verdana" size='2'>
<b>Para descifrar:</b> Hacemos la operación inversa restando la posición de la clave a la del criptograma. En ensamblador, si esa resta da un número negativo, le sumamos 95 para que vuelva a quedar en el rango positivo de 0 a 94, y finalmente le sumamos 32 para recuperar el carácter original.
</font></p>

<p align="justify"><font face="Verdana" size='2'>
<b>Caracteres de control:</b> Los tabuladores (ASCII 9), los saltos de línea (ASCII 10) y los retornos de carro (ASCII 13) los copiamos tal cual al archivo de salida sin cifrarlos y sin avanzar la posición de la clave. De esta manera, el archivo mantiene su estructura de párrafos y líneas original.
</font></p>

#### <font color='#000000' face="Courier New" size="4">2.1. Ejemplo de cifrado y descifrado.</font>

<p align="justify"><font face="Verdana" size='2'>
Si queremos cifrar la letra <code>'H'</code> (ASCII 72) con la letra de clave <code>'k'</code> (ASCII 107):
<ol>
  <li>Restamos 32 para normalizar: <code>Pos('H') = 72 − 32 = 40</code> y <code>Pos('k') = 107 − 32 = 75</code>.</li>
  <li>Sumamos y aplicamos módulo 95: <code>(40 + 75) mod 95 = 115 mod 95 = 20</code>.</li>
  <li>Sumamos 32 para volver a ASCII: <code>20 + 32 = 52</code>, que corresponde al carácter <code>'4'</code>.</li>
  <li>Para descifrar desde <code>'4'</code> (ASCII 52): <code>Pos('4') = 52 − 32 = 20</code>. Restamos la clave: <code>20 − 75 = −55</code>. Como es negativo, sumamos 95: <code>−55 + 95 = 40</code>. Sumamos 32: <code>40 + 32 = 72</code>, recuperando la letra original <code>'H'</code>.</li>
</ol>
</font></p>

### <font color='#000000' face="Courier New" size="5">3. Diseño del algoritmo.</font>

#### <font color='#000000' face="Courier New" size="4">3.1. Pseudocódigo del programa.</font>

<pre style="background-color:#F8F9F9; border:1px solid #D5D8DC; padding:15px; font-family:Courier New; font-size:13px;">
INICIO Menu_Principal:
    MIENTRAS sea verdadero HACER:
        Mostrar menu con opciones (1: Cifrar, 2: Descifrar, 3: Salir)
        Leer opcion del usuario
        
        SI opcion == 1 ENTONCES:
            read_archivo("input.txt")
            input_clave()
            procesar(modo = Cifrar)
            write_archivo("criptogram.txt")
            Mostrar mensaje de exito
            
        SINO SI opcion == 2 ENTONCES:
            read_archivo("criptogram.txt")
            input_clave()
            procesar(modo = Descifrar)
            write_archivo("deciphered.txt")
            Mostrar mensaje de exito
            
        SINO SI opcion == 3 ENTONCES:
            Mostrar mensaje "Saliendo del programa..."
            Terminar ejecucion (syscall 10)
            
        SINO:
            Mostrar mensaje de opcion invalida

SUBRUTINA read_archivo(nombre):
    Abrir archivo en modo solo lectura
    SI descriptor == -1 ENTONCES:
        Mostrar error de apertura y volver al menu
    Leer hasta 2049 bytes en buffer
    SI bytes_leidos == 2049 ENTONCES:
        Cerrar archivo, mostrar error de archivo grande y volver al menu
    SI bytes_leidos == 0 ENTONCES:
        Cerrar archivo, mostrar error de archivo vacio y volver al menu
    Cerrar archivo y volver

SUBRUTINA input_clave():
    Pedir y leer clave por consola (hasta 102 bytes)
    Contar caracteres hasta encontrar salto de linea, tabulador o nulo
    SI longitud == 0 ENTONCES:
        Mostrar error de clave vacia y volver al menu
    SI longitud >= 101 ENTONCES:
        Mostrar error de clave mayor a 100 caracteres y volver al menu
    Volver

SUBRUTINA procesar(modo):
    MIENTRAS no se termine el mensaje HACER:
        caracter = leer_byte_mensaje()
        SI caracter es tabulador (9), salto (10) o retorno (13) ENTONCES:
            Guardar caracter tal cual en output_buffer
            Avanzar solo puntero del mensaje
        SINO:
            letra_clave = leer_byte_clave()
            SI modo == Cifrar ENTONCES:
                resultado = ((caracter - 32) + (letra_clave - 32)) mod 95 + 32
            SINO:
                resultado = ((caracter - 32) - (letra_clave - 32) + 95) mod 95 + 32
            Guardar resultado en output_buffer
            Avanzar puntero del mensaje
            Avanzar puntero de la clave
            SI se acabo la clave ENTONCES:
                Reiniciar puntero de la clave al inicio (move $t0, $s2)
    Volver
</pre>

#### <font color='#000000' face="Courier New" size="4">3.2. Flujo general del programa.</font>

<p align="justify"><font face="Verdana" size='2'>
El programa arranca mostrando el menú por pantalla y pide un número entero con la <code>syscall 5</code>, que se guarda en el registro <code>&#36;s7</code>:
<ul>
  <li><b>Para cifrar (opción 1):</b> Va a <code>opt_cifrar</code>. Llama a <code>read_archivo</code> enviando como nombre <code>input.txt</code>. Si el archivo abre bien y tiene contenido válido, llama a <code>input_clave</code> para pedirle la clave al usuario. Luego prepara los punteros y llama a <code>procesar</code>. Al terminar de procesar, llama a <code>write_archivo</code> enviándole el nombre <code>criptogram.txt</code>, muestra el mensaje de éxito y regresa al menú.</li>
  <li><b>Para descifrar (opción 2):</b> Va a <code>opt_descifrar</code> y hace exactamente el mismo proceso modular, pero leyendo <code>criptogram.txt</code> y guardando el texto recuperado en <code>deciphered.txt</code>.</li>
  <li><b>Si ocurre algún error:</b> Si el archivo no existe, si está vacío, si supera los 2048 bytes, si la clave está vacía o si tiene más de 100 caracteres, el código salta al mensaje de error correspondiente, cierra el archivo si estaba abierto y regresa al menú para que el usuario pueda intentar de nuevo sin que se cierre el programa.</li>
  <li><b>Para salir (opción 3):</b> Muestra el mensaje de despedida y termina el programa con la <code>syscall 10</code>.</li>
</ul>
</font></p>

### <font color='#000000' face="Courier New" size="5">4. Organización de la implementación.</font>

<p align="justify"><font face="Verdana" size="2">
El código está organizado en las dos secciones clásicas de MIPS:
<ul>
  <li><b>Sección <code>.data</code>:</b> Aquí se reservan las cadenas con los nombres de los archivos (<code>input.txt</code>, <code>criptogram.txt</code> y <code>deciphered.txt</code>), el búfer de entrada de 2049 bytes, el de salida de 2048 bytes, los 102 bytes para la clave, y todos los textos que se imprimen en consola (menú, mensajes de éxito y mensajes de error).</li>
  <li><b>Sección <code>.text</code>:</b> Contiene todo el código dividido en bloques y subrutinas llamadas con <code>jal</code> y finalizadas con <code>jr &#36;ra</code> para reutilizar código.</li>
</ul>
</font></p>

#### <font color='#000000' face="Courier New" size="4">4.1. Registros utilizados.</font>

<p align="justify"><font face="Verdana" size="2">
En esta tabla resumimos qué papel cumple cada registro en nuestro programa:
</font></p>

<table style="width:100%; border-collapse:collapse; font-family:Verdana,sans-serif; font-size:14px;" border="1" cellpadding="9">
<thead>
<tr style="background-color:#E8F6F3; color:#0B5345;">
<th align="center" style="padding:10px; width:18%;">Registro</th>
<th align="center" style="padding:10px; width:22%;">Tipo de registro</th>
<th align="center" style="padding:10px;">¿Para qué lo usamos en el código?</th>
</tr>
</thead>
<tbody>
<tr>
<td><code>&#36;s0</code></td>
<td>Registro guardado</td>
<td>Contiene el identificador (descriptor) del archivo abierto, para leerlo, escribirlo y cerrarlo.</td>
</tr>
<tr>
<td><code>&#36;s1</code></td>
<td>Registro guardado</td>
<td>Contiene la cantidad de caracteres/bytes leídos del archivo (la longitud del texto).</td>
</tr>
<tr>
<td><code>&#36;s2</code></td>
<td>Registro guardado</td>
<td>Contiene la dirección base de la clave en memoria, para poder devolvernos al inicio de la clave cuando se reinicia.</td>
</tr>
<tr>
<td><code>&#36;s3</code></td>
<td>Registro guardado</td>
<td>Contiene la longitud calculada de la clave que escribió el usuario.</td>
</tr>
<tr>
<td><code>&#36;s5</code></td>
<td>Registro guardado</td>
<td>Puntero que apunta a la dirección en memoria del búfer de salida (<code>output_buffer</code>) donde vamos guardando el resultado.</td>
</tr>
<tr>
<td><code>&#36;s7</code></td>
<td>Registro guardado</td>
<td>Contiene la opción que eligió el usuario en el menú (1 para cifrar o 2 para descifrar).</td>
</tr>
<tr>
<td><code>&#36;t0</code></td>
<td>Temporal</td>
<td>Puntero a la letra actual de la clave que se está leyendo en ese momento.</td>
</tr>
<tr>
<td><code>&#36;t1</code></td>
<td>Temporal</td>
<td>Puntero al carácter actual del mensaje que se está leyendo del búfer de entrada.</td>
</tr>
<tr>
<td><code>&#36;t2, &#36;t3</code></td>
<td>Temporales</td>
<td>Guardan el byte de la clave y del mensaje respectivamente para compararlos y hacer las operaciones.</td>
</tr>
<tr>
<td><code>&#36;t4, &#36;t5</code></td>
<td>Temporales</td>
<td>Guardan los valores ya restados en 32 (valores entre 0 y 94).</td>
</tr>
<tr>
<td><code>&#36;t6</code></td>
<td>Temporal</td>
<td>Resultado de la suma o resta modular antes y después de sumarle 32.</td>
</tr>
<tr>
<td><code>&#36;t7</code></td>
<td>Temporal</td>
<td>Registro para hacer comparaciones con <code>slt</code>, <code>slti</code> o guardar valores constantes para comparar.</td>
</tr>
<tr>
<td><code>&#36;t8</code></td>
<td>Temporal</td>
<td>Contador de caracteres procesados de la clave actual (se compara con <code>&#36;s3</code> para reiniciar la clave).</td>
</tr>
<tr>
<td><code>&#36;t9</code></td>
<td>Temporal</td>
<td>Contador de caracteres procesados del mensaje (se compara con <code>&#36;s1</code> para saber cuándo parar).</td>
</tr>
<tr>
<td><code>&#36;a0 - &#36;a2</code></td>
<td>Argumentos</td>
<td>Pasan los datos a las llamadas al sistema (nombres de archivos, buffers, número de bytes).</td>
</tr>
<tr>
<td><code>&#36;v0</code></td>
<td>Valor de retorno / Syscall</td>
<td>Número del servicio del sistema a pedir y donde se reciben los resultados (como descriptores o bytes leídos).</td>
</tr>
<tr>
<td><code>&#36;ra</code></td>
<td>Dirección de retorno</td>
<td>Guarda la dirección de retorno para volver con <code>jr &#36;ra</code> cuando termina una subrutina.</td>
</tr>
</tbody>
</table>

#### <font color='#000000' face="Courier New" size="4">4.2. Procedimientos implementados.</font>

<table style="width:100%; border-collapse:collapse; font-family:Verdana,sans-serif; font-size:14px;" border="1" cellpadding="9">
<thead>
<tr style="background-color:#E8F6F3; color:#0B5345;">
<th align="center" style="padding:10px; width:18%;">Procedimiento</th>
<th align="center" style="padding:10px; width:22%;">Entradas</th>
<th align="center" style="padding:10px; width:22%;">Salidas</th>
<th align="center" style="padding:10px;">¿Qué hace?</th>
</tr>
</thead>
<tbody>
<tr>
<td><code>read_archivo</code></td>
<td><code>&#36;a0</code> con la dirección del nombre del archivo.</td>
<td><code>&#36;s1</code> con la cantidad de bytes leídos y llena el <code>buffer</code>.</td>
<td>Abre el archivo en solo lectura, revisa si hubo error al abrir (-1), lee el contenido, revisa si está vacío (0 bytes) o si supera los 2048 bytes, y cierra el archivo.</td>
</tr>
<tr>
<td><code>input_clave</code></td>
<td>Texto ingresado por el usuario en consola.</td>
<td><code>&#36;s2</code> con la dirección base y <code>&#36;s3</code> con su longitud.</td>
<td>Pide la clave por consola, calcula cuántas letras tiene hasta el salto de línea o tabulador, y comprueba que no esté vacía ni tenga más de 100 caracteres.</td>
</tr>
<tr>
<td><code>procesar</code></td>
<td>Registros <code>&#36;s1</code>, <code>&#36;s2</code>, <code>&#36;s3</code> y <code>&#36;s7</code>.</td>
<td>Llena <code>output_buffer</code> con los bytes listos.</td>
<td>Recorre el mensaje letra por letra. Si encuentra saltos o tabuladores los deja pasar directo. Si son caracteres imprimibles, los cifra o descifra según <code>&#36;s7</code>, y cuando se acaba la clave la reinicia al inicio.</td>
</tr>
<tr>
<td><code>write_archivo</code></td>
<td><code>&#36;a0</code> con el nombre de salida y <code>&#36;t9</code> con los bytes a escribir.</td>
<td>Guarda el archivo en el disco.</td>
<td>Crea y abre el archivo para escribir, escribe todo el <code>output_buffer</code> de un solo golpe, comprueba que se hayan escrito todos los bytes y cierra el archivo.</td>
</tr>
<tr>
<td><code>Manejadores de error</code></td>
<td>Saltos cuando ocurre alguna condición de error.</td>
<td>Mensaje en pantalla y retorno al menú.</td>
<td>Bloques específicos (<code>opcion_invalida</code>, <code>error_apertura</code>, <code>buffer_lleno</code>, <code>archivo_vacio</code>, <code>clave_vacia</code>, <code>clave_max</code>, <code>error_escritura</code>) que avisan del problema, cierran el archivo si estaba abierto y regresan al menú sin que el programa falle.</td>
</tr>
</tbody>
</table>

### <font color='#000000' face="Courier New" size="5">5. Decisiones de diseño.</font>

<p align="justify"><font face="Verdana" size="2">
Durante el desarrollo tomamos varias decisiones prácticas para que el programa fuera robusto, eficiente y fácil de usar. Aquí las explicamos y justificamos:
</font></p>

<table style="width:100%; border-collapse:collapse; font-family:Verdana,sans-serif; font-size:14px;" border="1" cellpadding="9">
<thead>
<tr style="background-color:#E8F6F3; color:#0B5345;">
<th align="center" style="padding:10px; width:22%;">Aspecto de diseño</th>
<th align="center" style="padding:10px; width:35%;">Decisión que tomamos en el código</th>
<th align="center" style="padding:10px;">¿Por qué lo hicimos así? (Justificación)</th>
</tr>
</thead>
<tbody>
<tr>
<td valign="top"><b>Alfabeto ASCII (32 al 126, mod 95)</b></td>
<td valign="top">Usamos todo el rango de caracteres imprimibles ASCII, desde el espacio (32) hasta la virgulilla <code>~</code> (126), con módulo 95.</td>
<td valign="top">El ejemplo clásico de Vigenère solo usa letras minúsculas (a-z), pero en un archivo en inglés real hay mayúsculas, números, signos de puntuación y espacios. Con este rango podemos cifrar el texto completo sin perder información ni ignorar caracteres.</td>
</tr>
<tr>
<td valign="top"><b>Caracteres de control (9, 10 y 13)</b></td>
<td valign="top">Los tabuladores (9), saltos de línea (10) y retornos de carro (13) se copian directamente sin cifrar y sin avanzar la clave.</td>
<td valign="top">Esto permite que el archivo mantenga su estructura de párrafos y tabulaciones intacta. Si los cifráramos con módulo 95, se convertirían en letras o símbolos y el texto perdería sus renglones al descifrarse.</td>
</tr>
<tr>
<td valign="top"><b>Comprobación de archivo grande con 2049 bytes</b></td>
<td valign="top">Reservamos <code>buffer: .space 2049</code> y le pedimos a la syscall 14 leer hasta 2049 bytes.</td>
<td valign="top">La guía dice que el tamaño máximo es 2048 bytes. Al pedir 2049, si el archivo tiene 2048 o menos se lee completo sin problema; pero si devuelve 2049, sabemos con certeza que el archivo supera el límite, lo cerramos y mostramos el error.</td>
</tr>
<tr>
<td valign="top"><b>Optimización en el reinicio de clave con <code>move</code></b></td>
<td valign="top">Guardamos la dirección base de la clave en <code>&#36;s2</code> y en la etiqueta <code>reset</code> hacemos <code>move &#36;t0, &#36;s2</code> en lugar de <code>la &#36;t0, clave</code>.</td>
<td valign="top">La instrucción <code>la</code> es una pseudoinstrucción que el ensamblador traduce a 2 instrucciones reales (<code>lui</code> y <code>ori</code>), mientras que <code>move</code> se traduce en 1 sola instrucción (<code>addu</code>). Como el reinicio se repite muchas veces durante el cifrado, ahorramos instrucciones y mejoramos el rendimiento.</td>
</tr>
<tr>
<td valign="top"><b>Uso de un búfer de salida intermedio</b></td>
<td valign="top">Guardamos todo el resultado procesado en <code>output_buffer</code> en memoria antes de escribirlo al disco.</td>
<td valign="top">Escribir carácter por carácter en el archivo implicaría llamar a la syscall 15 miles de veces, lo que sería muy lento. Con el búfer intermedio se hace una sola escritura para todo el archivo.</td>
</tr>
<tr>
<td valign="top"><b>Validación estricta de escritura</b></td>
<td valign="top">Al escribir el archivo con la syscall 15 comprobamos que los bytes escritos sean iguales a los procesados con <code>bne &#36;v0, &#36;t9, error_escritura</code>.</td>
<td valign="top">Nos asegura que el archivo se haya escrito por completo en el disco y que no haya quedado a medias o cortado.</td>
</tr>
<tr>
<td valign="top"><b>Cierre de archivos en situaciones de error</b></td>
<td valign="top">En las etiquetas de error de archivo vacío, buffer lleno y error de escritura, cerramos el archivo con syscall 16 antes de volver al menú.</td>
<td valign="top">Si no se cierra el descriptor, el sistema operativo se queda con archivos abiertos ocupando memoria, lo que podría provocar fallos si el usuario sigue usando el menú.</td>
</tr>
</tbody>
</table>

### <font color='#000000' face="Courier New" size="6">6. Pruebas y resultados.</font>

<p align="justify"><font face="Verdana" size="2">
Probamos el programa directamente en MARS cubriendo tanto el funcionamiento normal como todos los posibles errores. Todos los casos funcionaron como esperábamos:
</font></p>

<table style="width:100%; border-collapse:collapse; font-family:Verdana,sans-serif; font-size:14px;" border="1" cellpadding="9">
<thead>
<tr style="background-color:#E8F6F3; color:#0B5345;">
<th align="center" style="padding:10px; width:15%;">Prueba</th>
<th align="center" style="padding:10px; width:22%;">Entrada / Clave</th>
<th align="center" style="padding:10px; width:28%;">Resultado esperado</th>
<th align="center" style="padding:10px; width:25%;">Resultado observado en MARS</th>
<th align="center" style="padding:10px; width:10%;">Estado</th>
</tr>
</thead>
<tbody>
<tr>
<td><b>Caso básico</b></td>
<td><code>smartphone</code> con clave <code>cipher</code></td>
<td>Cifrar el texto y luego recuperarlo intacto al descifrar.</td>
<td>Cifra correctamente y al descifrar recupera <code>smartphone</code>.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Texto con mayúsculas y números</b></td>
<td><code>Hello World! 123</code> con clave <code>segura</code></td>
<td>Cifrar mayúsculas, espacios y números sin corromper el contenido.</td>
<td>El archivo <code>deciphered.txt</code> recupera exactamente <code>Hello World! 123</code>.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Texto con varias líneas y tabulaciones</b></td>
<td>Texto de varias líneas con saltos (<code>\n</code>) y tabuladores (<code>\t</code>)</td>
<td>Mantener la estructura y renglones del texto original.</td>
<td>El archivo descifrado mantiene exactamente los mismos saltos de línea y tabuladores.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Opción de menú inválida</b></td>
<td>Ingresar <code>0</code> o <code>99</code> en el menú</td>
<td>Mostrar mensaje de error y volver a pedir la opción.</td>
<td>Muestra <code>[Error]: Opcion invalida. Intenta de nuevo.</code> y recarga el menú.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Archivo que no existe</b></td>
<td>Pedir cifrar cuando no existe <code>input.txt</code></td>
<td>Detectar descriptor -1 y avisar al usuario.</td>
<td>Muestra <code>[Error]: no se pudo abrir el archivo.</code> y regresa al menú.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Archivo vacío</b></td>
<td><code>input.txt</code> con 0 bytes</td>
<td>Detectar 0 bytes leídos, cerrar archivo y avisar.</td>
<td>Muestra <code>[Error]: el archivo esta vacio.</code> y vuelve al menú.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Archivo mayor a 2048 bytes</b></td>
<td><code>input.txt</code> con 2100 caracteres</td>
<td>Detectar lectura de 2049 bytes y rechazar el archivo.</td>
<td>Muestra <code>[Error]: el archivo supera los 2048 bytes.</code> y vuelve al menú.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Clave vacía</b></td>
<td>Presionar solo ENTER al pedir la clave</td>
<td>Detectar longitud 0 y rechazarla.</td>
<td>Muestra <code>[Error]: la clave no puede estar vacia.</code> y vuelve al menú.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Clave mayor a 100 caracteres</b></td>
<td>Ingresar una clave de 101 caracteres</td>
<td>Detectar el exceso de caracteres y rechazarla.</td>
<td>Muestra <code>[Error]: la clave no puede superar los 100 caracteres.</code> y vuelve al menú.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
<tr>
<td><b>Salida del programa</b></td>
<td>Opción <code>3</code> en el menú</td>
<td>Despedirse y terminar la ejecución limpiamente.</td>
<td>Muestra <code>Saliendo del programa...</code> y acaba con syscall 10.</td>
<td align="center"><font color="green"><b>Superada</b></font></td>
</tr>
</tbody>
</table>

### <font color='#000000' face="Courier New" size="5">7. Aspectos técnicos y ajustes durante el desarrollo.</font>

<p align="justify"><font face="Verdana" size="2">
Durante el desarrollo y las pruebas en MARS revisamos varios puntos importantes del programa y los dejamos listos:
</font></p>

<ul>
  <li align="justify"><font face="Verdana" size="2">[<b>X</b>] <b>Control de saltos y menú:</b> Revisamos que cada opción del menú llame a su respectiva rutina (<code>opt_cifrar</code>, <code>opt_descifrar</code> o <code>exit</code>) y que la rutina <code>procesar</code> use el registro <code>&#36;s7</code> para saber si debe sumar o restar.</font></li>
  <li align="justify"><font face="Verdana" size="2">[<b>X</b>] <b>Manejo de errores:</b> Se implementó el control para archivo no encontrado, archivo vacío, archivo mayor a 2048 bytes, clave vacía y clave mayor a 100 caracteres, asegurando que el programa no se caiga ante entradas incorrectas.</font></li>
  <li align="justify"><font face="Verdana" size="2">[<b>X</b>] <b>Uso de registros:</b> Se organizó el uso de los registros <code>&#36;s0 - &#36;s7</code> para que cada uno tuviera una función clara, eliminando registros repetidos y optimizando el reinicio de la clave con <code>move &#36;t0, &#36;s2</code>.</font></li>
  <li align="justify"><font face="Verdana" size="2">[<b>X</b>] <b>Límites de memoria:</b> Ajustamos los tamaños de los búferes con un byte de más (2049 para el archivo y 102 para la clave) para poder detectar con precisión cuándo se pasa del límite permitido.</font></li>
</ul>

#### <font color='#000000' face="Courier New" size="4">7.1. Retos encontrados al depurar en MARS.</font>

<p align="justify"><font face="Verdana" size="2">
<ul>
  <li><b>Detalle con la Syscall 8 para leer la clave:</b> En MARS, la syscall 8 siempre guarda el último byte del tamaño para poner el terminador nulo <code>\0</code>. Si poníamos tamaño 101, solo leía 100 caracteres y nunca llegaba a 101 para detectar el error. Por eso le dimos tamaño 102 en el buffer y en la syscall; así, si el usuario escribe 101 caracteres, el programa alcanza a leer el carácter 101 y puede avisar que se pasó del límite.</li>
  <li><b>Restas negativas al descifrar:</b> Al descifrar en MIPS, si la posición del mensaje es menor que la de la clave, la resta da un número negativo. Para resolverlo sin hacer divisiones pesadas, usamos <code>slt &#36;t7, &#36;t6, &#36;zero</code>; si da menor que cero, simplemente le sumamos 95 para devolverlo al rango positivo y luego le sumamos 32.</li>
</ul>
</font></p>

### <font color='#000000' face="Courier New" size="5">8. Observaciones y conclusiones.</font>

#### <font color='#000000' face="Courier New" size="4">8.1. Observaciones.</font>

<p align="justify"><font face="Verdana" size="2">
<ul>
  <li>Programar en ensamblador MIPS requiere estar muy pendiente de la memoria y de los registros. A diferencia de un lenguaje de alto nivel donde uno no se preocupa por cómo se guardan las cadenas, aquí hay que controlar cada byte, los saltos de línea y los terminadores nulos.</li>
  <li>Modularizar el código con <code>jal</code> y <code>jr &#36;ra</code> facilitó mucho el trabajo. Poder reutilizar funciones como <code>read_archivo</code> y <code>write_archivo</code> tanto para el cifrado como para el descifrado ahorró bastantes líneas y evitó repetir código.</li>
  <li>Probar el código en MARS y revisar los registros paso a paso nos ayudó a encontrar detalles importantes, como el uso de <code>move</code> en vez de <code>la</code> entre registros y la forma en que MARS maneja los límites de lectura de cadenas.</li>
</ul>
</font></p>

#### <font color='#000000' face="Courier New" size="4">8.2. Conclusiones.</font>

<p align="justify"><font face="Verdana" size="2">
<ul>
  <li>Logramos implementar de forma completa y funcional el cifrado y descifrado de Vigenère en ensamblador MIPS de 32 bits, verificando que el texto se recupera idéntico al descifrarlo.</li>
  <li>La decisión de usar el rango ASCII de 95 caracteres (32 al 126) y dejar pasar los caracteres de control (tabuladores y saltos de línea) fue muy práctica, porque permite que el programa funcione con archivos de texto reales sin dañar sus renglones ni perder caracteres especiales.</li>
  <li>El manejo de errores hace que el programa sea mucho más confiable ante entradas incorrectas del usuario o problemas con los archivos, avisando de lo ocurrido y volviendo al menú de forma segura.</li>
  <li>Pudimos aplicar en la práctica los conceptos vistos en clase: el uso de registros temporales y guardados, modos de direccionamiento, llamadas al sistema mediante <i>syscalls</i> y la importancia de reducir instrucciones de máquina en bucles para que el código sea más eficiente.</li>
</ul>
</font></p>